In [1]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [2]:
pip install pyproj


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Archivos de excesos de velocidad

In [3]:
fecha_ini = '20260501'
fecha_fin = '20260531'

mes = 'may'

In [4]:
#Importar archivos de F3 - Zonal

carpeta_origen = 'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Zonal/Excepciones _coord_fb'

dataframes = [] # Lista para almacenar cada archivo

fecha_inicio = f'{fecha_ini}' 
fecha_fin = f'{fecha_fin}'

for archivo in os.listdir(carpeta_origen):
    if archivo.endswith('_coord_fb.csv'):  # Filtra solo los archivos con extensión .csv
        ruta_archivo = os.path.join(carpeta_origen, archivo)
        df = pd.read_csv(ruta_archivo, encoding='latin')
        
        # Obtener la fecha del nombre del archivo
        fecha_archivo = pd.to_datetime(archivo.split('_')[0], format='%Y%m%d', errors='coerce')

        # Filtrar por fechas
        if fecha_archivo >= pd.to_datetime(fecha_inicio, format='%Y%m%d') and fecha_archivo <= pd.to_datetime(fecha_fin, format='%Y%m%d'):
            dataframes.append(df)  # Agregar el DataFrame a la lista

# Combinar todos los DataFrames en uno solo
if dataframes:
    excepciones = pd.concat(dataframes)

else:
    print("No se encontraron archivos que cumplan con el filtro de fechas.")
    
excepciones.head()

,Fecha,Concesión,Concesionario de Operación,Código Bus,N° FMS Bus,Instante,Excepción,Descripción de la Excepción,Id línea,Línea,...,Estado de Localización,Valor Aceleración,Coordenada X,Coordenada Y,Id Viaje,Tipo de Nodo,Id Nodo,Valor Tiempo,Límite de Velocidad,Exceso de Velocidad
0,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 5:46,7,Aceleracion excesiva o Frenado brusco,10551.0,KL307,...,5,-12,515794,597601,2,NaN,NaN,NaN,NaN,NaN
1,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:22,7,Aceleracion excesiva o Frenado brusco,10551.0,KL307,...,5,-13,507358,602768,2,NaN,NaN,NaN,NaN,NaN
2,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:24,7,Aceleracion excesiva o Frenado brusco,10551.0,KL307,...,5,-10,507089,602690,2,NaN,NaN,NaN,NaN,NaN
3,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:38,7,Aceleracion excesiva o Frenado brusco,10551.0,KL307,...,5,-21,505325,602010,2,NaN,NaN,NaN,NaN,NaN
4,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:39,7,Aceleracion excesiva o Frenado brusco,10551.0,KL307,...,5,9,505372,602395,2,NaN,NaN,NaN,NaN,NaN


In [5]:
excepciones['Id línea'] = excepciones['Id línea'].replace('', 0).fillna(0).astype(int)
excepciones['Tabla'] = excepciones['Tabla'].replace('', 0).fillna(0).astype(int)
excepciones['Excepción'] = excepciones['Excepción'].replace('', 0).fillna(0).astype(int)
excepciones['N° FMS Bus'] = excepciones['N° FMS Bus'].replace('', 0).fillna(0).astype(int)
excepciones['Ruta'] = excepciones['Ruta'].replace('', 0).fillna(0).astype(int)
excepciones['Límite de Velocidad'] = excepciones['Límite de Velocidad'].replace('', 0).fillna(0).astype(int)
excepciones['Código Conductor'] = excepciones['Código Conductor'].replace('', 0).fillna(0).astype(int)

excepciones.head()

,Fecha,Concesión,Concesionario de Operación,Código Bus,N° FMS Bus,Instante,Excepción,Descripción de la Excepción,Id línea,Línea,...,Estado de Localización,Valor Aceleración,Coordenada X,Coordenada Y,Id Viaje,Tipo de Nodo,Id Nodo,Valor Tiempo,Límite de Velocidad,Exceso de Velocidad
0,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 5:46,7,Aceleracion excesiva o Frenado brusco,10551,KL307,...,5,-12,515794,597601,2,NaN,NaN,NaN,0,NaN
1,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:22,7,Aceleracion excesiva o Frenado brusco,10551,KL307,...,5,-13,507358,602768,2,NaN,NaN,NaN,0,NaN
2,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:24,7,Aceleracion excesiva o Frenado brusco,10551,KL307,...,5,-10,507089,602690,2,NaN,NaN,NaN,0,NaN
3,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:38,7,Aceleracion excesiva o Frenado brusco,10551,KL307,...,5,-21,505325,602010,2,NaN,NaN,NaN,0,NaN
4,1/05/2026,ENGATIVA ZN,GMOVIL ENGATIVA,Z50-2002,502002,1/05/2026 6:39,7,Aceleracion excesiva o Frenado brusco,10551,KL307,...,5,9,505372,602395,2,NaN,NaN,NaN,0,NaN


In [6]:
excepciones = excepciones.drop(columns=['Descripción de la Excepción', 'Concesión', 'Concesionario de Operación'])

excepciones.head()

,Fecha,Código Bus,N° FMS Bus,Instante,Excepción,Id línea,Línea,Tabla,Ruta,Offset Ruta,...,Estado de Localización,Valor Aceleración,Coordenada X,Coordenada Y,Id Viaje,Tipo de Nodo,Id Nodo,Valor Tiempo,Límite de Velocidad,Exceso de Velocidad
0,1/05/2026,Z50-2002,502002,1/05/2026 5:46,7,10551,KL307,2,12856,8820.0,...,5,-12,515794,597601,2,NaN,NaN,NaN,0,NaN
1,1/05/2026,Z50-2002,502002,1/05/2026 6:22,7,10551,KL307,2,12856,20259.0,...,5,-13,507358,602768,2,NaN,NaN,NaN,0,NaN
2,1/05/2026,Z50-2002,502002,1/05/2026 6:24,7,10551,KL307,2,12856,20622.0,...,5,-10,507089,602690,2,NaN,NaN,NaN,0,NaN
3,1/05/2026,Z50-2002,502002,1/05/2026 6:38,7,10551,KL307,2,12856,24720.0,...,5,-21,505325,602010,2,NaN,NaN,NaN,0,NaN
4,1/05/2026,Z50-2002,502002,1/05/2026 6:39,7,10551,KL307,2,12856,25209.0,...,5,9,505372,602395,2,NaN,NaN,NaN,0,NaN


In [7]:
from pyproj import Transformer

# Crear un transformador de coordenadas de UTM Zona 18N (EPSG:32618) a WGS84 (EPSG:4326)
transformer = Transformer.from_crs("epsg:32618", "epsg:4326", always_xy=True)

# Aplicar la transformación a las columnas PosX (easting) y PosY (northing)
excepciones['Coord.longitude'], excepciones['Coord.latitude'] = transformer.transform(excepciones['Coordenada X'].values, excepciones['Coordenada Y'].values)

excepciones.head()

,Fecha,Código Bus,N° FMS Bus,Instante,Excepción,Id línea,Línea,Tabla,Ruta,Offset Ruta,...,Coordenada X,Coordenada Y,Id Viaje,Tipo de Nodo,Id Nodo,Valor Tiempo,Límite de Velocidad,Exceso de Velocidad,Coord.longitude,Coord.latitude
0,1/05/2026,Z50-2002,502002,1/05/2026 5:46,7,10551,KL307,2,12856,8820.0,...,515794,597601,2,NaN,NaN,NaN,0,NaN,-74.857433,5.406506
1,1/05/2026,Z50-2002,502002,1/05/2026 6:22,7,10551,KL307,2,12856,20259.0,...,507358,602768,2,NaN,NaN,NaN,0,NaN,-74.933577,5.453262
2,1/05/2026,Z50-2002,502002,1/05/2026 6:24,7,10551,KL307,2,12856,20622.0,...,507089,602690,2,NaN,NaN,NaN,0,NaN,-74.936005,5.452557
3,1/05/2026,Z50-2002,502002,1/05/2026 6:38,7,10551,KL307,2,12856,24720.0,...,505325,602010,2,NaN,NaN,NaN,0,NaN,-74.951930,5.446407
4,1/05/2026,Z50-2002,502002,1/05/2026 6:39,7,10551,KL307,2,12856,25209.0,...,505372,602395,2,NaN,NaN,NaN,0,NaN,-74.951505,5.449890


In [8]:
excepciones['Cantidad'] = 1

In [9]:
excepciones['Fecha'] = pd.to_datetime(excepciones['Fecha'], format='%d/%m/%Y')

# Directorio donde se guardarán los archivos
output_dir = 'C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco'

# Crear el directorio si no existe
os.makedirs(output_dir, exist_ok=True)

# Agrupar por fecha y exportar cada grupo como un archivo separado
for fecha, grupo in excepciones.groupby(excepciones['Fecha']):
    # Formatear la fecha como cadena (por ejemplo, 20240804)
    fecha_str = fecha.strftime('%Y%m%d')
    
    # Generar el nombre del archivo
    file_name = f'{fecha_str}_excepciones.csv'
    
    # Generar la ruta completa del archivo
    file_path = os.path.join(output_dir, file_name)
    
    # Exportar el grupo como un archivo CSV utilizando punto y coma como delimitador
    grupo.to_csv(file_path, sep=';', index=False)
    print(f'Archivo exportado: {file_path}')

<>:4: SyntaxWarning: invalid escape sequence '\J'
<>:4: SyntaxWarning: invalid escape sequence '\J'
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_16608\2722680723.py:4: SyntaxWarning: invalid escape sequence '\J'
  output_dir = 'C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco'


Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260501_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260502_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260503_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260504_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260505_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260506_excepciones.csv
Archivo exportado: C:/Users\Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco\20260507_excepciones.csv
Archivo exportado: C

In [10]:
resultado_resumido = (
    excepciones
    .groupby([
        'Fecha','N° FMS Bus','Servicio Bus','Id línea',
        'Línea','Ruta','Tabla','Código Conductor'
    ], as_index=False)
    .agg({
        'Cantidad': 'sum'
    })
)

In [11]:
resultado_resumido = resultado_resumido.rename(columns={
    'Cantidad': 'total_eventos'
})

resultado_resumido.head()

,Fecha,N° FMS Bus,Servicio Bus,Id línea,Línea,Ruta,Tabla,Código Conductor,total_eventos
0,2026-05-01,502002,CE1730002,10551,KL307,12856,2,505851,143
1,2026-05-01,502002,CE173G010,10551,KL307,12856,15,509332,63
2,2026-05-01,502003,CE1730001,10551,KL307,12856,1,509093,128
3,2026-05-01,502007,CE09EG005,10342,KB309,12400,15,510274,106
4,2026-05-01,502007,CE0D50004,10331,12,12475,4,509947,49


In [12]:
resultado_resumido.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/Frenado_brusco/{mes}_excepciones.csv',sep=';', index=False )